# Project Data Cleaning

This notebook documents the data inspection, cleaning, and verification process for the project.

The workflow follows four stages:

1. Anthropic data cleaning
2. BLS data cleaning
3. O*NET data cleaning
4. Data integration and merge verification

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from clean_anthropic import (
    load_anthropic,
    clean_anthropic,
    audit_anthropic,
)

## 1. Anthropic Economic Index

**Source file:** `aei_claude_ai_2026-06-26.csv`

The Anthropic dataset contains Claude.ai usage metrics across multiple
geographies, node types, hierarchy levels, metrics, and time periods.

The project uses detailed occupation-level records and selected AI-use
metrics relevant to the California workforce analysis.

In [3]:
anthropic_raw = load_anthropic(
    "aei_claude_ai_2026-06-26.csv"
)

anthropic_raw.shape

(1636573, 10)

In [4]:
anthropic_raw.head()

,date_start,date_end,geo_id,geo_level,category_name,hierarchy_level,metric_id,value,node_name,node_external_id
0,2026-05-01,2026-06-01,ZA-WC,subregion,onet,2,pct,0.25,"Prepare financial documents, reports, or budgets.",4.A.3.b.6.I01
1,2026-05-01,2026-06-01,ZA-WC,subregion,onet,0,pct,0.40,Locate unusual or unique information in respon...,22420
2,2026-05-01,2026-06-01,ZA-WC,subregion,onet,0,pct,0.68,"Answer customers' questions, and provide infor...",2408
3,2026-05-01,2026-06-01,ZA-KZN,subregion,onet,1,pct,0.94,Advise others on business or operational matters.,4.A.4.b.6.I05.D10
4,2026-05-01,2026-06-01,ZA-KZN,subregion,onet,1,pct,2.18,Tutor students who need extra assistance.,4.A.4.b.3.I02.D10


In [5]:
print(
    "Exact duplicate rows:",
    anthropic_raw.duplicated().sum()
)

anthropic_raw.isna().sum().to_frame(
    name="missing_count"
)

Exact duplicate rows: 0


,missing_count
date_start,0
date_end,0
geo_id,0
geo_level,0
category_name,0
hierarchy_level,0
metric_id,0
value,0
node_name,0
node_external_id,0


### Node categories and hierarchy levels

The source contains multiple node categories and hierarchy levels.
These are inspected before applying the occupation-level filter.

In [6]:
anthropic_raw[
    "category_name"
].value_counts(dropna=False)

category_name
onet              744544
request           465202
soc_occupation    353817
overall            73010
Name: count, dtype: int64

In [7]:
(
    anthropic_raw
    .groupby(
        ["category_name", "hierarchy_level"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
)

category_name   hierarchy_level
onet            0                  340069
request         0                  204398
onet            3                  200876
soc_occupation  1                  179557
                0                  174260
request         2                  172196
onet            1                  134864
request         1                   88608
overall         0                   73010
onet            2                   68735
dtype: int64

### Geographic levels

The dataset contains observations at multiple geographic levels.
The project uses selected global occupation measures and
California-specific usage-share records.

In [8]:
anthropic_raw[
    "geo_level"
].value_counts(dropna=False)

geo_level
country      673272
global       554941
subregion    408360
Name: count, dtype: int64

In [9]:
anthropic_raw[
    "metric_id"
].value_counts(dropna=False)

metric_id
pct                                         532491
use_case_work_pct                            22061
multitasking_pct                             22061
human_education_years_mean                   22061
use_case_coursework_pct                      22061
use_case_personal_pct                        22061
human_only_ability_pct                       22060
ai_autonomy_mean                             22060
ai_education_years_mean                      22060
artifact_idea_or_brainstorm_pct              22059
artifact_presentation_or_slides_pct          22059
artifact_sql_or_database_query_pct           22059
artifact_plan_or_strategy_pct                22059
artifact_math_or_calculation_pct             22059
artifact_other_pct                           22059
artifact_blog_or_article_pct                 22059
artifact_email_or_message_pct                22059
artifact_audio_or_music_pct                  22059
artifact_explanation_or_answer_pct           22059
artifact_image_or_gra

In [10]:
(
    anthropic_raw[
        ["date_start", "date_end"]
    ]
    .drop_duplicates()
    .sort_values(
        ["date_start", "date_end"]
    )
)

,date_start,date_end
236,2026-04-01,2026-05-01
0,2026-05-01,2026-06-01


### Cleaning rules

The following rules were defined before cleaning:

1. Keep only detailed occupation records relevant to the project.
2. Keep only the geography and metrics used in the project.
3. Keep `node_external_id` as text to preserve the full O*NET-SOC code.

In [11]:
anthropic_clean = clean_anthropic(
    anthropic_raw
)

anthropic_clean.shape

(5121, 10)

In [12]:
anthropic_clean.head()

,date_start,date_end,geo_id,geo_level,category_name,hierarchy_level,metric_id,value,node_name,node_external_id
0,2026-05-01,2026-06-01,GLOBAL,global,soc_occupation,0,ai_autonomy_mean,2.47,Recreational Vehicle Service Technicians,49-3092.00
1,2026-05-01,2026-06-01,GLOBAL,global,soc_occupation,0,ai_autonomy_mean,2.59,Gambling and Sports Book Writers and Runners,39-3012.00
2,2026-05-01,2026-06-01,GLOBAL,global,soc_occupation,0,ai_autonomy_mean,2.72,Occupational Therapists,29-1122.00
3,2026-05-01,2026-06-01,GLOBAL,global,soc_occupation,0,ai_autonomy_mean,2.82,"Multiple Machine Tool Setters, Operators, and ...",51-4081.00
4,2026-05-01,2026-06-01,GLOBAL,global,soc_occupation,0,ai_autonomy_mean,2.65,"Library Science Teachers, Postsecondary",25-1082.00


In [13]:
anthropic_audit = audit_anthropic(
    anthropic_raw,
    anthropic_clean
)

### Row-count reconciliation

The source is first reduced to detailed occupation records and then
to the geography and metric combinations required for the project.

In [14]:
anthropic_row_summary = pd.DataFrame({
    "Stage": [
        "Raw Anthropic data",
        "Detailed occupation records",
        "Project-relevant records",
    ],
    "Rows": [
        anthropic_audit["raw_rows"],
        anthropic_audit["occupation_rows"],
        anthropic_audit["cleaned_rows"],
    ],
})

anthropic_row_summary

,Stage,Rows
0,Raw Anthropic data,1636573
1,Detailed occupation records,174260
2,Project-relevant records,5121


In [15]:
(
    anthropic_clean[
        ["category_name", "hierarchy_level"]
    ]
    .value_counts()
    .to_frame(name="rows")
)

,,rows
category_name,hierarchy_level,
soc_occupation,0,5121


In [16]:
print(
    "Invalid SOC codes:",
    anthropic_audit["invalid_soc_codes"]
)

Invalid SOC codes: 0


### Retained geography and metric combinations

The cleaned data should contain only the global occupation measures
and California usage-share measure used in the project.

In [17]:
anthropic_audit[
    "metric_combinations"
]

,geo_level,geo_id,metric_id,rows
0,global,GLOBAL,ai_autonomy_mean,1414
1,global,GLOBAL,collaboration_bucket_augmentation_pct,1411
2,global,GLOBAL,collaboration_bucket_automation_pct,1411
3,subregion,US-CA,pct,885


In [18]:
print(
    "node_external_id dtype:",
    anthropic_clean[
        "node_external_id"
    ].dtype
)

anthropic_clean[
    ["node_external_id", "node_name"]
].drop_duplicates().head(10)

node_external_id dtype: string


,node_external_id,node_name
0,49-3092.00,Recreational Vehicle Service Technicians
1,39-3012.00,Gambling and Sports Book Writers and Runners
2,29-1122.00,Occupational Therapists
3,51-4081.00,"Multiple Machine Tool Setters, Operators, and ..."
4,25-1082.00,"Library Science Teachers, Postsecondary"
5,17-2121.00,Marine Engineers and Naval Architects
6,39-9032.00,Recreation Workers
7,13-1071.00,Human Resources Specialists
8,17-3026.01,Nanotechnology Engineering Technologists and T...
9,19-2012.00,Physicists


In [19]:
pd.DataFrame({
    "Check": [
        "Exact duplicate rows",
        "Repeated candidate keys",
    ],
    "Count": [
        anthropic_audit[
            "exact_duplicates"
        ],
        anthropic_audit[
            "repeated_candidate_keys"
        ],
    ],
})

,Check,Count
0,Exact duplicate rows,0
1,Repeated candidate keys,0


In [20]:
(
    anthropic_audit[
        "missing_values"
    ]
    .rename("missing_count")
    .to_frame()
)

,missing_count
date_start,0
date_end,0
geo_id,0
geo_level,0
category_name,0
hierarchy_level,0
metric_id,0
value,0
node_name,0
node_external_id,0


No missing values were found in the retained Anthropic records.
Therefore, no missing-value imputation or deletion was required.

In [21]:
anthropic_audit[
    "rows_by_period"
]

,date_start,date_end,geo_level,metric_id,rows
0,2026-04-01,2026-05-01,global,ai_autonomy_mean,696
1,2026-04-01,2026-05-01,global,collaboration_bucket_augmentation_pct,695
2,2026-04-01,2026-05-01,global,collaboration_bucket_automation_pct,695
3,2026-04-01,2026-05-01,subregion,pct,425
4,2026-05-01,2026-06-01,global,ai_autonomy_mean,718
5,2026-05-01,2026-06-01,global,collaboration_bucket_augmentation_pct,716
6,2026-05-01,2026-06-01,global,collaboration_bucket_automation_pct,716
7,2026-05-01,2026-06-01,subregion,pct,460


### Anthropic cleaning summary

The Anthropic source was filtered from the full multi-level dataset
to detailed SOC occupation records and then to the geography and
metric combinations required for the project.

Verification confirmed that:

- only the intended detailed occupation records remain;
- only the intended global and California metric combinations remain;
- full O*NET-SOC occupation codes were preserved;
- no invalid retained occupation codes were found;
- no exact duplicate rows were found;
- no repeated candidate keys were found; and
- no missing values were found in the retained subset.

## 2. BLS OEWS Data

**Source files:**

- `MSA_M2025_dl.xlsx`
- `BOS_M2025_dl.xlsx`

The BLS OEWS data provide employment and wage estimates by geographic
area and occupation.

The project uses California metropolitan and nonmetropolitan areas
and detailed occupation records.

In [22]:
from clean_bls import (
    load_bls,
    clean_bls,
    audit_bls,
)

In [23]:
msa_raw, nonmetro_raw = load_bls(
    "MSA_M2025_dl.xlsx",
    "BOS_M2025_dl.xlsx",
)

print("MSA shape:", msa_raw.shape)
print("Nonmetro shape:", nonmetro_raw.shape)

MSA shape: (150023, 32)
Nonmetro shape: (48689, 32)


### Original data structure

The metropolitan and nonmetropolitan files use the same OEWS schema.
Both contain records for multiple states and occupation aggregation levels.

In [24]:
msa_raw.head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.32,29.92,45.19,24900,30980,44340,62230,93990,NaN,NaN
1,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,45.9,64.34,92.98,47790,64990,95480,133840,193390,NaN,NaN
2,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.7,113.51,187.68,88400,121980,165770,236100,390370,NaN,NaN
3,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,41.44,67.13,101.14,43770,57740,86200,139630,210360,NaN,NaN
4,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,...,52.45,74.4,95.07,71010,85770,109100,154750,197740,NaN,NaN


In [25]:
nonmetro_raw.head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,0100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,19.71,26.99,36.87,23570,30750,41000,56130,76700,NaN,NaN
1,0100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,46.11,61.34,84.82,50920,65010,95910,127590,176420,NaN,NaN
2,0100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,72.82,87.54,110.05,123440,128860,151460,182090,228910,NaN,NaN
3,0100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,50,75.01,102.7,51990,64780,103990,156020,213610,NaN,NaN
4,0100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-2022,Sales Managers,...,62.39,83.96,104.94,65860,87490,129770,174630,218270,NaN,NaN


### Geographic and occupation scope

Before cleaning, the BLS files are inspected to confirm that they contain
multiple states and multiple occupation aggregation levels.

In [26]:
msa_raw["PRIM_STATE"].value_counts(dropna=False).head(20)

PRIM_STATE
CA    11449
TX     9984
FL     8957
PA     6414
NC     6027
NY     5592
MI     5273
OH     5263
WI     4649
GA     4599
TN     4083
IN     4022
WA     3889
AL     3801
LA     3492
VA     3448
MO     3267
SC     3138
OR     3104
CO     3083
Name: count, dtype: int64

In [27]:
nonmetro_raw["PRIM_STATE"].value_counts(dropna=False).head(20)

PRIM_STATE
TX    2256
OH    1696
MI    1608
PA    1597
GA    1565
WI    1530
IL    1514
IA    1499
KY    1481
MS    1469
NC    1468
OK    1465
MN    1437
TN    1418
MO    1394
AR    1310
NY    1290
VA    1264
IN    1259
AL    1197
Name: count, dtype: int64

In [28]:
msa_raw["PRIM_STATE"].value_counts(dropna=False).head(20)

PRIM_STATE
CA    11449
TX     9984
FL     8957
PA     6414
NC     6027
NY     5592
MI     5273
OH     5263
WI     4649
GA     4599
TN     4083
IN     4022
WA     3889
AL     3801
LA     3492
VA     3448
MO     3267
SC     3138
OR     3104
CO     3083
Name: count, dtype: int64

In [29]:
nonmetro_raw["O_GROUP"].value_counts(dropna=False)

O_GROUP
detailed    45542
major        3010
total         137
Name: count, dtype: int64

In [30]:
msa_ca = msa_raw[
    msa_raw["PRIM_STATE"] == "CA"
].copy()

nonmetro_ca = nonmetro_raw[
    nonmetro_raw["PRIM_STATE"] == "CA"
].copy()

print("California MSA rows:", len(msa_ca))
print("California nonmetro rows:", len(nonmetro_ca))

California MSA rows: 11449
California nonmetro rows: 1042


In [31]:
print("California MSA occupation groups:")
print(msa_ca["O_GROUP"].value_counts())

print("\nCalifornia nonmetro occupation groups:")
print(nonmetro_ca["O_GROUP"].value_counts())

California MSA occupation groups:
O_GROUP
detailed    10874
major         550
total          25
Name: count, dtype: int64

California nonmetro occupation groups:
O_GROUP
detailed    973
major        66
total         3
Name: count, dtype: int64


### BLS special values

Several employment and wage fields contain BLS publication symbols
instead of numeric values.

These symbols must be identified before numeric conversion because they
do not represent zero.

In [32]:
key_numeric_cols = [
    "TOT_EMP",
    "H_MEAN",
    "A_MEAN",
    "H_MEDIAN",
    "A_MEDIAN",
]

for col in key_numeric_cols:
    print(f"\n{col} - MSA")
    print(
        msa_raw[col]
        .astype(str)
        .value_counts()
        .loc[lambda x: x.index.isin(["*", "**", "#"])]
    )

    print(f"{col} - Nonmetro")
    print(
        nonmetro_raw[col]
        .astype(str)
        .value_counts()
        .loc[lambda x: x.index.isin(["*", "**", "#"])]
    )


TOT_EMP - MSA
TOT_EMP
**    2939
Name: count, dtype: int64
TOT_EMP - Nonmetro
TOT_EMP
**    760
Name: count, dtype: int64

H_MEAN - MSA
H_MEAN
*    9291
#     207
Name: count, dtype: int64
H_MEAN - Nonmetro
H_MEAN
*    2952
#      39
Name: count, dtype: int64

A_MEAN - MSA
A_MEAN
*    1714
#     207
Name: count, dtype: int64
A_MEAN - Nonmetro
A_MEAN
*    435
#     39
Name: count, dtype: int64

H_MEDIAN - MSA
H_MEDIAN
*    9291
#     208
Name: count, dtype: int64
H_MEDIAN - Nonmetro
H_MEDIAN
*    2952
#      39
Name: count, dtype: int64

A_MEDIAN - MSA
A_MEDIAN
*    1714
#     208
Name: count, dtype: int64
A_MEDIAN - Nonmetro
A_MEDIAN
*    435
#     39
Name: count, dtype: int64


### Cleaning rules

The following rules were defined before cleaning:

1. Keep only California records.
2. Keep only `O_GROUP = detailed`.
3. Combine metro and nonmetro records after adding an area-type indicator.
4. Convert numeric fields only after handling BLS special symbols.
5. Treat `*` and `**` as missing, not zero.
6. Preserve `#` as a flagged high-wage value rather than zero.
7. Keep plausible extreme values unless there is evidence they are invalid.

In [33]:
bls_clean = clean_bls(
    msa_raw,
    nonmetro_raw,
)

bls_clean.shape

(11847, 90)

In [34]:
bls_clean.head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,A_PCT25_high_wage_flag,A_MEDIAN_raw,A_MEDIAN_unavailable_flag,A_MEDIAN_high_wage_flag,A_PCT75_raw,A_PCT75_unavailable_flag,A_PCT75_high_wage_flag,A_PCT90_raw,A_PCT90_unavailable_flag,A_PCT90_high_wage_flag
0,12540,"Bakersfield-Delano, CA",4,CA,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,False,195400,False,False,287210,False,False,386830,False,False
1,12540,"Bakersfield-Delano, CA",4,CA,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,False,104460,False,False,162260,False,False,218740,False,False
2,12540,"Bakersfield-Delano, CA",4,CA,0,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,...,False,149960,False,False,199640,False,False,271050,False,False
3,12540,"Bakersfield-Delano, CA",4,CA,0,Cross-industry,cross-industry,1235,11-2022,Sales Managers,...,False,98270,False,False,155440,False,False,211000,False,False
4,12540,"Bakersfield-Delano, CA",4,CA,0,Cross-industry,cross-industry,1235,11-2032,Public Relations Managers,...,False,137200,False,False,177390,False,False,207170,False,False


In [35]:
bls_audit = audit_bls(
    msa_raw,
    nonmetro_raw,
    bls_clean,
)

### Row-count reconciliation

The row counts below show the progression from the two original OEWS files
to the California detailed-occupation subset used in the project.

In [36]:
bls_row_summary = pd.DataFrame({
    "Stage": [
        "Raw MSA data",
        "Raw nonmetro data",
        "California detailed MSA",
        "California detailed nonmetro",
        "Combined cleaned BLS data",
    ],
    "Rows": [
        bls_audit["msa_raw_rows"],
        bls_audit["nonmetro_raw_rows"],
        bls_audit["msa_project_rows"],
        bls_audit["nonmetro_project_rows"],
        bls_audit["cleaned_rows"],
    ],
})

bls_row_summary

,Stage,Rows
0,Raw MSA data,150023
1,Raw nonmetro data,48689
2,California detailed MSA,10874
3,California detailed nonmetro,973
4,Combined cleaned BLS data,11847


In [37]:
bls_audit["states_remaining"]

['CA']

In [38]:
bls_audit["occupation_groups_remaining"]

['detailed']

### Area-type verification

The two BLS source files are combined only after labeling each row as
metropolitan or nonmetropolitan so that the original geographic type
remains identifiable.

In [39]:
bls_audit["area_type_counts"]

,area_type,rows
0,metro,10874
1,nonmetro,973


In [40]:
pd.DataFrame({
    "Measure": [
        "Metro areas",
        "Nonmetro areas",
        "Total OEWS areas",
        "Unique occupations",
    ],
    "Count": [
        bls_audit["metro_areas"],
        bls_audit["nonmetro_areas"],
        bls_audit["total_areas"],
        bls_audit["unique_occupations"],
    ],
})

,Measure,Count
0,Metro areas,25
1,Nonmetro areas,3
2,Total OEWS areas,28
3,Unique occupations,783


### Area × occupation key check

The expected analytical grain is one record per OEWS area × occupation.
The combination of `AREA` and `OCC_CODE` is checked for repeated keys.

In [41]:
pd.DataFrame({
    "Check": [
        "Exact duplicate rows",
        "Repeated AREA + OCC_CODE keys",
    ],
    "Count": [
        bls_audit["exact_duplicates"],
        bls_audit["repeated_area_occupation_keys"],
    ],
})

,Check,Count
0,Exact duplicate rows,0
1,Repeated AREA + OCC_CODE keys,0


### Special-symbol counts before conversion

The table below records the number of `*`, `**`, and `#` values in the
California detailed-occupation records before numeric conversion.

In [42]:
special_counts = bls_audit["special_symbol_counts"]

special_counts[
    (
        special_counts["*"]
        + special_counts["**"]
        + special_counts["#"]
    ) > 0
]

,area_type,column,*,**,#
0,metro,TOT_EMP,0,201,0
1,metro,EMP_PRSE,0,201,0
2,metro,JOBS_1000,0,201,0
3,metro,LOC_QUOTIENT,0,201,0
6,metro,H_MEAN,686,0,13
7,metro,A_MEAN,146,0,13
8,metro,MEAN_PRSE,122,0,0
9,metro,H_PCT10,686,0,13
10,metro,H_PCT25,686,0,13
11,metro,H_MEDIAN,686,0,13


### Unavailable-value verification

BLS `*` and `**` symbols are treated as unavailable values rather than
zero. The original symbol counts should equal the number of corresponding
unavailable-value flags in the cleaned data.

In [43]:
flag_check = bls_audit["flag_check"]

flag_check[
    flag_check[
        "raw_unavailable_symbols"
    ] > 0
][
    [
        "column",
        "raw_unavailable_symbols",
        "unavailable_flags",
    ]
]

,column,raw_unavailable_symbols,unavailable_flags
0,TOT_EMP,215,215
1,EMP_PRSE,215,215
2,JOBS_1000,215,215
3,LOC_QUOTIENT,215,215
6,H_MEAN,732,732
7,A_MEAN,154,154
8,MEAN_PRSE,129,129
9,H_PCT10,732,732
10,H_PCT25,732,732
11,H_MEDIAN,732,732


### High-wage flag verification

BLS `#` values represent wages at or above the publication threshold.
They are not interpreted as zero. Instead, the numeric value is treated
as unavailable and a high-wage flag is retained.

In [44]:
flag_check[
    flag_check[
        "raw_high_wage_symbols"
    ] > 0
][
    [
        "column",
        "raw_high_wage_symbols",
        "high_wage_flags",
    ]
]

,column,raw_high_wage_symbols,high_wage_flags
6,H_MEAN,15,15
7,A_MEAN,15,15
9,H_PCT10,15,15
10,H_PCT25,15,15
11,H_MEDIAN,15,15
12,H_PCT75,15,15
13,H_PCT90,15,15
14,A_PCT10,15,15
15,A_PCT25,15,15
16,A_MEDIAN,15,15


### Numeric conversion verification

Only recognized BLS special symbols should become missing during
conversion. Any other value that fails numeric conversion is counted
as an unexpected conversion failure.

In [45]:
bls_audit[
    "conversion_failures"
]

,column,failed_conversions
0,TOT_EMP,0
1,EMP_PRSE,0
2,JOBS_1000,0
3,LOC_QUOTIENT,0
4,PCT_TOTAL,0
5,PCT_RPT,0
6,H_MEAN,0
7,A_MEAN,0
8,MEAN_PRSE,0
9,H_PCT10,0


In [46]:
[
    (col, bls_clean[col].dtype)
    for col in [
        "TOT_EMP",
        "A_MEAN",
        "A_MEDIAN",
        "H_MEAN",
        "H_MEDIAN",
    ]
]

[('TOT_EMP', dtype('float64')),
 ('A_MEAN', dtype('float64')),
 ('A_MEDIAN', dtype('float64')),
 ('H_MEAN', dtype('float64')),
 ('H_MEDIAN', dtype('float64'))]

### Missing values after cleaning

Missing values in the cleaned BLS measures primarily represent
unavailable or threshold-coded values from the source, rather than
values imputed or deleted during cleaning.

In [47]:
bls_missing = bls_audit[
    "missing_values"
]

bls_missing[
    bls_missing["missing_count"] > 0
]

,missing_count
TOT_EMP,215
EMP_PRSE,215
JOBS_1000,215
LOC_QUOTIENT,215
PCT_TOTAL,11847
PCT_RPT,11847
H_MEAN,747
A_MEAN,169
MEAN_PRSE,129
H_PCT10,747


### Suspicious and extreme values

Extreme published values are reviewed but are not automatically removed.
Unusual values are retained unless there is evidence that they are invalid.

In [48]:
bls_audit[
    "numeric_summary"
][
    ["count", "mean", "min", "50%", "max"]
]

,count,mean,min,50%,max
TOT_EMP,11632.0,1485.343019,30.000,230.000,385530.000
EMP_PRSE,11632.0,11.747017,0.000,9.400,49.900
JOBS_1000,11632.0,2.272463,0.005,0.832,143.928
LOC_QUOTIENT,11632.0,1.349537,0.030,0.920,84.300
PCT_TOTAL,0.0,NaN,NaN,NaN,NaN
PCT_RPT,0.0,NaN,NaN,NaN,NaN
H_MEAN,11100.0,38.738381,16.610,32.505,280.980
A_MEAN,11678.0,82101.442028,34540.000,69520.000,584430.000
MEAN_PRSE,11718.0,3.312852,0.000,2.500,29.800
H_PCT10,11100.0,24.990777,16.500,21.620,114.930


In [49]:
bls_clean[
    [
        "AREA_TITLE",
        "OCC_CODE",
        "OCC_TITLE",
        "A_MEDIAN",
        "A_MEDIAN_high_wage_flag",
    ]
].sort_values(
    "A_MEDIAN",
    ascending=False
).head(10)

,AREA_TITLE,OCC_CODE,OCC_TITLE,A_MEDIAN,A_MEDIAN_high_wage_flag
7579,"San Jose-Sunnyvale-Santa Clara, CA",29-1241,"Ophthalmologists, Except Pediatric",700850.0,False
239,"Bakersfield-Delano, CA",33-1021,First-Line Supervisors of Firefighting and Pre...,632190.0,False
3624,"Oxnard-Thousand Oaks-Ventura, CA",29-1249,"Surgeons, All Other",535800.0,False
6215,"San Diego-Chula Vista-Carlsbad, CA",29-1212,Cardiologists,493670.0,False
2041,"Los Angeles-Long Beach-Anaheim, CA",29-1224,Radiologists,488670.0,False
2043,"Los Angeles-Long Beach-Anaheim, CA",29-1241,"Ophthalmologists, Except Pediatric",433520.0,False
8414,"Santa Cruz-Watsonville, CA",29-1216,General Internal Medicine Physicians,430970.0,False
6919,"San Francisco-Oakland-Fremont, CA",29-1241,"Ophthalmologists, Except Pediatric",429080.0,False
9221,"Santa Rosa-Petaluma, CA",29-1229,"Physicians, All Other",421490.0,False
5696,"Salinas, CA",29-1215,Family Medicine Physicians,410370.0,False


In [50]:
bls_clean[
    [
        "AREA_TITLE",
        "OCC_CODE",
        "OCC_TITLE",
        "TOT_EMP",
    ]
].sort_values(
    "TOT_EMP",
    ascending=False
).head(10)

,AREA_TITLE,OCC_CODE,OCC_TITLE,TOT_EMP
2074,"Los Angeles-Long Beach-Anaheim, CA",31-1120,Home Health and Personal Care Aides,385530.0
2122,"Los Angeles-Long Beach-Anaheim, CA",35-3023,Fast Food and Counter Workers,160270.0
2175,"Los Angeles-Long Beach-Anaheim, CA",41-2031,Retail Salespersons,130650.0
2471,"Los Angeles-Long Beach-Anaheim, CA",53-7062,"Laborers and Freight, Stock, and Material Move...",120720.0
6947,"San Francisco-Oakland-Fremont, CA",31-1120,Home Health and Personal Care Aides,119120.0
2240,"Los Angeles-Long Beach-Anaheim, CA",43-9061,"Office Clerks, General",112270.0
2171,"Los Angeles-Long Beach-Anaheim, CA",41-2011,Cashiers,109940.0
2028,"Los Angeles-Long Beach-Anaheim, CA",29-1141,Registered Nurses,109360.0
1725,"Los Angeles-Long Beach-Anaheim, CA",11-1021,General and Operations Managers,104610.0
4529,"Riverside-San Bernardino-Ontario, CA",31-1120,Home Health and Personal Care Aides,102760.0


In [51]:
pd.DataFrame({
    "Variable": [
        "TOT_EMP",
        "A_MEDIAN",
    ],
    "Missing": [
        bls_clean["TOT_EMP"].isna().sum(),
        bls_clean["A_MEDIAN"].isna().sum(),
    ],
    "Available": [
        bls_clean["TOT_EMP"].notna().sum(),
        bls_clean["A_MEDIAN"].notna().sum(),
    ],
})

,Variable,Missing,Available
0,TOT_EMP,215,11632
1,A_MEDIAN,169,11678


### BLS cleaning summary

The BLS metropolitan and nonmetropolitan OEWS files were filtered to
California detailed occupations and then combined into one dataset while
preserving the original area type.

BLS publication symbols were handled before numeric conversion:

- `*` and `**` were treated as unavailable rather than zero;
- `#` values were retained through explicit high-wage flags;
- valid published values were converted to numeric form.

The final BLS dataset contains 11,847 California area × occupation
records across 25 metropolitan areas and 3 nonmetropolitan areas.
No repeated `AREA + OCC_CODE` keys were found.

## 3. O*NET Occupation Data

**Source file:** `Occupation Data.xlsx`

The O*NET occupation table provides standardized occupation codes,
titles, and descriptions.

This table is primarily used as an occupation-reference table to
harmonize occupation identifiers across Anthropic and BLS.

In [52]:
from clean_onet import (
    load_onet,
    clean_onet,
    audit_onet,
)

In [53]:
onet_raw = load_onet(
    "Occupation Data.xlsx"
)

onet_raw.shape

(1016, 3)

In [54]:
onet_raw.head()

,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."


### Original-data checks

The O*NET table is inspected for missing values, exact duplicates,
and repeated occupation identifiers before any transformation.

In [55]:
print(
    "Exact duplicate rows:",
    onet_raw.duplicated().sum()
)

print(
    "Duplicate O*NET-SOC codes:",
    onet_raw["O*NET-SOC Code"].duplicated().sum()
)

Exact duplicate rows: 0
Duplicate O*NET-SOC codes: 0


In [56]:
onet_raw.isna().sum().to_frame(
    name="missing_count"
)

,missing_count
O*NET-SOC Code,0
Title,0
Description,0


In [57]:
onet_raw[
    ["O*NET-SOC Code", "Title"]
].head(10)

,O*NET-SOC Code,Title
0,11-1011.00,Chief Executives
1,11-1011.03,Chief Sustainability Officers
2,11-1021.00,General and Operations Managers
3,11-1031.00,Legislators
4,11-2011.00,Advertising and Promotions Managers
5,11-2021.00,Marketing Managers
6,11-2022.00,Sales Managers
7,11-2032.00,Public Relations Managers
8,11-2033.00,Fundraising Managers
9,11-3012.00,Administrative Services Managers


### Cleaning rules

The following rules were defined before cleaning:

1. Keep the full `O*NET-SOC Code` as text.
2. Derive a separate six-digit SOC code for BLS matching.
3. Do not deduplicate solely on the derived six-digit SOC code.

In [58]:
onet_clean = clean_onet(
    onet_raw
)

onet_clean.shape

(1016, 4)

In [59]:
onet_clean.head()

,O*NET-SOC Code,Title,Description,SOC6
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,11-1011
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",11-1011
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",11-1021
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes...",11-1031
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",11-2011


In [60]:
onet_audit = audit_onet(
    onet_raw,
    onet_clean
)

### Row-count reconciliation

No occupation records are intentionally removed from the O*NET table.
Cleaning preserves the original rows and adds a derived six-digit SOC key.

In [61]:
pd.DataFrame({
    "Stage": [
        "Raw O*NET data",
        "Cleaned O*NET data",
    ],
    "Rows": [
        onet_audit["raw_rows"],
        onet_audit["cleaned_rows"],
    ],
})

,Stage,Rows
0,Raw O*NET data,1016
1,Cleaned O*NET data,1016


In [62]:
print(
    "Unique full O*NET codes:",
    onet_audit["unique_full_codes"]
)

print(
    "Duplicate full O*NET codes:",
    onet_audit["duplicate_full_codes"]
)

print(
    "Invalid full O*NET codes:",
    onet_audit["invalid_full_codes"]
)

Unique full O*NET codes: 1016
Duplicate full O*NET codes: 0
Invalid full O*NET codes: 0


### Six-digit SOC derivation

A separate `SOC6` field is derived from the full O*NET-SOC code for
matching with BLS occupation codes.

The original full O*NET-SOC code is retained unchanged.

In [63]:
onet_clean[
    [
        "O*NET-SOC Code",
        "SOC6",
        "Title",
    ]
].head(10)

,O*NET-SOC Code,SOC6,Title
0,11-1011.00,11-1011,Chief Executives
1,11-1011.03,11-1011,Chief Sustainability Officers
2,11-1021.00,11-1021,General and Operations Managers
3,11-1031.00,11-1031,Legislators
4,11-2011.00,11-2011,Advertising and Promotions Managers
5,11-2021.00,11-2021,Marketing Managers
6,11-2022.00,11-2022,Sales Managers
7,11-2032.00,11-2032,Public Relations Managers
8,11-2033.00,11-2033,Fundraising Managers
9,11-3012.00,11-3012,Administrative Services Managers


In [64]:
print(
    "Unique SOC6 codes:",
    onet_audit["unique_soc6_codes"]
)

print(
    "Invalid SOC6 codes:",
    onet_audit["invalid_soc6_codes"]
)

Unique SOC6 codes: 867
Invalid SOC6 codes: 0


### Repeated six-digit SOC codes

Repeated `SOC6` values are not treated as duplicates.

Different O*NET occupations can share the same six-digit SOC base
through different O*NET extensions, so these records are retained
for later merge review.

In [65]:
pd.DataFrame({
    "Measure": [
        "Rows belonging to repeated SOC6 groups",
        "Repeated SOC6 codes",
    ],
    "Count": [
        onet_audit["repeated_soc6_rows"],
        onet_audit["repeated_soc6_codes"],
    ],
})

,Measure,Count
0,Rows belonging to repeated SOC6 groups,225
1,Repeated SOC6 codes,76


In [66]:
onet_audit[
    "repeated_soc6_detail"
].head(30)

,SOC6,O*NET-SOC Code,Title
0,11-1011,11-1011.00,Chief Executives
1,11-1011,11-1011.03,Chief Sustainability Officers
2,11-3013,11-3013.00,Facilities Managers
3,11-3013,11-3013.01,Security Managers
4,11-3031,11-3031.00,Financial Managers
5,11-3031,11-3031.01,Treasurers and Controllers
6,11-3031,11-3031.03,Investment Fund Managers
7,11-3051,11-3051.00,Industrial Production Managers
8,11-3051,11-3051.01,Quality Control Systems Managers
9,11-3051,11-3051.02,Geothermal Production Managers


In [67]:
pd.DataFrame({
    "Check": [
        "Exact duplicate rows",
        "Duplicate full O*NET-SOC codes",
    ],
    "Count": [
        onet_audit["exact_duplicates"],
        onet_audit["duplicate_full_codes"],
    ],
})

,Check,Count
0,Exact duplicate rows,0
1,Duplicate full O*NET-SOC codes,0


In [68]:
onet_audit[
    "missing_values"
]

,missing_count
O*NET-SOC Code,0
Title,0
Description,0
SOC6,0


No missing values were found in the O*NET occupation table.
Therefore, no missing-value imputation or deletion was required.

### O*NET cleaning summary

The O*NET occupation table required minimal cleaning.

The original full O*NET-SOC codes were preserved as text, and a separate
six-digit SOC key was derived for later BLS matching.

Verification confirmed that:

- the original row count was preserved;
- full O*NET-SOC codes remained unique;
- no invalid full occupation codes were found;
- derived six-digit SOC codes follow the expected format;
- repeated six-digit SOC values were retained rather than incorrectly
  treated as duplicates; and
- no missing values were found.

## 4. Data Integration and Merge Verification

The cleaned datasets are integrated using occupation identifiers.

Three merge steps are used:

1. Anthropic → O*NET using the full O*NET-SOC code.
2. BLS → O*NET using the six-digit SOC code and the O*NET `.00` base occupation.
3. BLS → Anthropic using the harmonized six-digit SOC code.

Merge relationships, unmatched keys, row counts, and duplicate keys are
checked after each step.

In [69]:
from merge_data import (
    merge_datasets,
    audit_merge,
)

### Merge plan

| Merge | Left key | Right key | Expected relationship |
|---|---|---|---|
| Anthropic → O*NET | `node_external_id` | `O*NET-SOC Code` | many-to-one |
| BLS → O*NET | `OCC_CODE` | `SOC6` for `.00` occupations | many-to-one |
| BLS → Anthropic | `OCC_CODE` | `SOC6` | many-to-many |

The Anthropic-to-O*NET merge uses the full O*NET-SOC code because the two
sources use the same detailed occupation identifier.

The BLS-to-O*NET merge uses only O*NET `.00` base occupations to prevent
one BLS occupation from matching multiple O*NET extensions.

The final BLS-to-Anthropic merge is expected to increase the number of rows
because Anthropic remains in long format with multiple metrics and periods
per occupation.

In [70]:
(
    anthropic_onet,
    bls_onet,
    integrated_long,
) = merge_datasets(
    anthropic_clean,
    bls_clean,
    onet_clean,
)

In [71]:
print("Anthropic + O*NET:", anthropic_onet.shape)
print("BLS + O*NET:", bls_onet.shape)
print("Integrated long table:", integrated_long.shape)

Anthropic + O*NET: (5121, 15)
BLS + O*NET: (11847, 95)
Integrated long table: (67194, 105)


In [72]:
merge_audit = audit_merge(
    anthropic_clean,
    bls_clean,
    anthropic_onet,
    bls_onet,
    integrated_long,
)

### Row-count reconciliation

Reference-table joins should not increase the number of source records.

The final BLS-to-Anthropic join is expected to increase the row count
because each BLS area × occupation record can match multiple Anthropic
month × metric observations.

In [73]:
merge_audit["row_counts"]

,Stage,Rows
0,Clean Anthropic,5121
1,Anthropic after O*NET merge,5121
2,Clean BLS,11847
3,BLS after O*NET merge,11847
4,Integrated long table,67194


### Anthropic → O*NET

Anthropic occupation records are matched to O*NET using the full occupation
identifier:

`node_external_id = O*NET-SOC Code`

The expected relationship is many Anthropic observations to one O*NET
occupation.

In [74]:
merge_audit[
    "anthropic_match_counts"
]

,status,rows
0,both,5121
1,left_only,0
2,right_only,0


In [75]:
merge_audit[
    "anthropic_unmatched"
]

,node_external_id,node_name


In [76]:
print(
    "Unexpected row expansion:",
    merge_audit[
        "anthropic_onet_row_expansion"
    ]
)

Unexpected row expansion: 0


### BLS → O*NET

BLS uses six-digit occupation codes, while O*NET includes additional
occupation extensions.

Only O*NET `.00` base occupations are used for direct BLS matching in order
to avoid one-to-many row expansion.

In [77]:
merge_audit[
    "bls_match_counts"
]

,status,rows
0,both,11561
1,left_only,286
2,right_only,0


In [78]:
merge_audit[
    "bls_unmatched"
]

,OCC_CODE,OCC_TITLE
0,13-1020,Buyers and Purchasing Agents
1,13-2020,Property Appraisers and Assessors
2,21-1018,"Substance Abuse, Behavioral Disorder, and Ment..."
3,25-2052,"Special Education Teachers, Kindergarten and E..."
4,25-9045,"Teaching Assistants, Except Postsecondary"
5,29-2010,Clinical Laboratory Technologists and Technicians
6,31-1120,Home Health and Personal Care Aides
7,39-7010,Tour and Travel Guides
8,47-4090,Miscellaneous Construction and Related Workers
9,51-2028,"Electrical, Electronic, and Electromechanical ..."


In [79]:
print(
    "Unique unmatched BLS occupations:",
    len(
        merge_audit[
            "bls_unmatched"
        ]
    )
)

Unique unmatched BLS occupations: 12


In [80]:
print(
    "Unexpected row expansion:",
    merge_audit[
        "bls_onet_row_expansion"
    ]
)

Unexpected row expansion: 0


In [81]:
print(
    "Repeated AREA + OCC_CODE keys after O*NET merge:",
    merge_audit[
        "bls_key_duplicates_after_onet"
    ]
)

Repeated AREA + OCC_CODE keys after O*NET merge: 0


### BLS → Anthropic

The BLS occupation-area records are matched to Anthropic occupation-level
observations using the harmonized six-digit SOC code.

`OCC_CODE` repeats across geographic areas in BLS, while `SOC6` repeats
across periods and metrics in Anthropic. Therefore, the merge is
many-to-many on the occupation key.

The resulting row expansion is expected. Correctness is verified using
the full integrated grain:

`AREA × OCC_CODE × period × geography × metric`.


In [82]:
merge_audit[
    "bls_anthropic_match_counts"
]

,status,rows
0,both,64267
1,left_only,2927
2,right_only,0


In [83]:
merge_audit[
    "bls_without_anthropic"
].head(20)

,AREA,AREA_TITLE,OCC_CODE,OCC_TITLE
0,12540,"Bakersfield-Delano, CA",11-9039,"Education Administrators, All Other"
1,12540,"Bakersfield-Delano, CA",11-9199,"Managers, All Other"
2,12540,"Bakersfield-Delano, CA",13-1020,Buyers and Purchasing Agents
3,12540,"Bakersfield-Delano, CA",13-1199,"Business Operations Specialists, All Other"
4,12540,"Bakersfield-Delano, CA",13-2020,Property Appraisers and Assessors
5,12540,"Bakersfield-Delano, CA",13-2099,"Financial Specialists, All Other"
6,12540,"Bakersfield-Delano, CA",15-1299,"Computer Occupations, All Other"
7,12540,"Bakersfield-Delano, CA",17-2199,"Engineers, All Other"
8,12540,"Bakersfield-Delano, CA",17-3019,"Drafters, All Other"
9,12540,"Bakersfield-Delano, CA",17-3021,Aerospace Engineering and Operations Technolog...


In [84]:
print(
    "BLS area × occupation records without Anthropic matches:",
    len(
        merge_audit[
            "bls_without_anthropic"
        ]
    )
)

BLS area × occupation records without Anthropic matches: 2927


### Integrated-table grain

At this stage, the integrated table remains in long format.

A populated Anthropic record is expected to represent:

**one OEWS area × occupation × period × geography × metric**

This is intentionally more detailed than the final dashboard feature mart.
Aggregation or pivoting of the Anthropic measures is deferred to later
feature construction.

In [85]:
print(
    "Repeated integrated keys:",
    merge_audit[
        "integrated_duplicate_keys"
    ]
)

Repeated integrated keys: 0


In [86]:
integrated_long[
    [
        "AREA",
        "AREA_TITLE",
        "area_type",
        "OCC_CODE",
        "OCC_TITLE",
        "TOT_EMP",
        "A_MEDIAN",
        "date_start",
        "geo_level",
        "metric_id",
        "value",
    ]
].head(20)

,AREA,AREA_TITLE,area_type,OCC_CODE,OCC_TITLE,TOT_EMP,A_MEDIAN,date_start,geo_level,metric_id,value
0,12540,"Bakersfield-Delano, CA",metro,11-1011,Chief Executives,390.0,195400.0,2026-05-01,global,ai_autonomy_mean,2.91
1,12540,"Bakersfield-Delano, CA",metro,11-1011,Chief Executives,390.0,195400.0,2026-04-01,global,collaboration_bucket_augmentation_pct,47.36
2,12540,"Bakersfield-Delano, CA",metro,11-1011,Chief Executives,390.0,195400.0,2026-04-01,global,collaboration_bucket_automation_pct,52.64
3,12540,"Bakersfield-Delano, CA",metro,11-1011,Chief Executives,390.0,195400.0,2026-04-01,global,ai_autonomy_mean,2.87
4,12540,"Bakersfield-Delano, CA",metro,11-1011,Chief Executives,390.0,195400.0,2026-05-01,global,collaboration_bucket_automation_pct,53.32
5,12540,"Bakersfield-Delano, CA",metro,11-1011,Chief Executives,390.0,195400.0,2026-05-01,global,collaboration_bucket_augmentation_pct,46.68
6,12540,"Bakersfield-Delano, CA",metro,11-1011,Chief Executives,390.0,195400.0,2026-05-01,subregion,pct,0.16
7,12540,"Bakersfield-Delano, CA",metro,11-1011,Chief Executives,390.0,195400.0,2026-04-01,subregion,pct,0.17
8,12540,"Bakersfield-Delano, CA",metro,11-1021,General and Operations Managers,4100.0,104460.0,2026-05-01,global,ai_autonomy_mean,2.97
9,12540,"Bakersfield-Delano, CA",metro,11-1021,General and Operations Managers,4100.0,104460.0,2026-05-01,global,collaboration_bucket_automation_pct,44.39


### Merge verification summary

The source tables were joined using explicitly harmonized occupation keys.

Verification checked:

- the expected relationship for each merge;
- matched and unmatched occupation codes;
- row counts before and after each merge;
- unexpected row expansion in the O*NET reference joins;
- preservation of the BLS area × occupation key; and
- uniqueness of the expected long-format integrated key.

Unmatched occupations were retained rather than forced into uncertain
matches.

## 5. Final Validation

The final cleaning checks reconcile the main datasets and confirm that
the outputs preserve the intended data structure.

In [87]:
final_dataset_summary = pd.DataFrame({
    "Dataset": [
        "Anthropic cleaned",
        "BLS cleaned",
        "O*NET cleaned",
        "Anthropic + O*NET",
        "BLS + O*NET",
        "Integrated long",
    ],
    "Rows": [
        len(anthropic_clean),
        len(bls_clean),
        len(onet_clean),
        len(anthropic_onet),
        len(bls_onet),
        len(integrated_long),
    ],
})

final_dataset_summary

,Dataset,Rows
0,Anthropic cleaned,5121
1,BLS cleaned,11847
2,O*NET cleaned,1016
3,Anthropic + O*NET,5121
4,BLS + O*NET,11847
5,Integrated long,67194


In [88]:
duplicate_summary = pd.DataFrame({
    "Dataset / key": [
        "Anthropic exact rows",
        "Anthropic candidate key",
        "BLS exact rows",
        "BLS AREA + OCC_CODE",
        "O*NET exact rows",
        "O*NET full code",
        "Integrated long key",
    ],
    "Duplicate count": [
        anthropic_audit["exact_duplicates"],
        anthropic_audit["repeated_candidate_keys"],
        bls_audit["exact_duplicates"],
        bls_audit["repeated_area_occupation_keys"],
        onet_audit["exact_duplicates"],
        onet_audit["duplicate_full_codes"],
        merge_audit["integrated_duplicate_keys"],
    ],
})

duplicate_summary

,Dataset / key,Duplicate count
0,Anthropic exact rows,0
1,Anthropic candidate key,0
2,BLS exact rows,0
3,BLS AREA + OCC_CODE,0
4,O*NET exact rows,0
5,O*NET full code,0
6,Integrated long key,0


In [89]:
metric_record_counts = pd.DataFrame({
    "Metric": [
        "BLS total employment",
        "BLS annual median wage",
        "Anthropic AI metric values",
    ],
    "Available records": [
        bls_clean["TOT_EMP"].notna().sum(),
        bls_clean["A_MEDIAN"].notna().sum(),
        anthropic_clean["value"].notna().sum(),
    ],
    "Missing records": [
        bls_clean["TOT_EMP"].isna().sum(),
        bls_clean["A_MEDIAN"].isna().sum(),
        anthropic_clean["value"].isna().sum(),
    ],
})

metric_record_counts

,Metric,Available records,Missing records
0,BLS total employment,11632,215
1,BLS annual median wage,11678,169
2,Anthropic AI metric values,5121,0


## 6. Individual Record Checks

Five records are checked manually, including difficult cases.

For each case, the expected result is defined from the source value and
cleaning rule before the cleaned output is inspected.


### Expected results defined before checking output

| Check | Source record | Expected result | Reason |
|---|---|---|---|
| 1 | Anthropic `49-3092.00`, May 2026, global `ai_autonomy_mean` | Record remains unchanged with value `2.47` | It is a valid detailed occupation record and an allowed global project metric. |
| 2 | BLS Bakersfield `13-2082` Tax Preparers, `TOT_EMP = **` | `TOT_EMP` becomes missing and `TOT_EMP_unavailable_flag = True` | `**` means the employment estimate is unavailable, not zero. |
| 3 | BLS Bakersfield `29-1214` Emergency Medicine Physicians, `A_MEDIAN = #` | `A_MEDIAN` becomes missing and `A_MEDIAN_high_wage_flag = True` | `#` represents a wage at or above the BLS publication threshold. |
| 4 | O*NET `11-1011.03` Chief Sustainability Officers | Full code remains `11-1011.03`; derived `SOC6 = 11-1011` | The extension must be preserved while a separate six-digit key is created for BLS matching. |
| 5 | BLS Bakersfield `13-1020` Buyers and Purchasing Agents | Record remains in BLS and is marked `left_only` in the O*NET direct match | No direct `.00` O*NET base occupation is available, so the code should not be forced into a match. |


In [90]:
record_checks = []

# 1. Normal Anthropic record
anthropic_original = anthropic_raw[
    (anthropic_raw["node_external_id"] == "49-3092.00")
    & (anthropic_raw["date_start"] == "2026-05-01")
    & (anthropic_raw["geo_level"] == "global")
    & (anthropic_raw["metric_id"] == "ai_autonomy_mean")
].iloc[0]

anthropic_actual = anthropic_clean[
    (anthropic_clean["node_external_id"] == "49-3092.00")
    & (anthropic_clean["date_start"] == "2026-05-01")
    & (anthropic_clean["geo_level"] == "global")
    & (anthropic_clean["metric_id"] == "ai_autonomy_mean")
].iloc[0]

record_checks.append({
    "Check": 1,
    "Record": "Anthropic 49-3092.00, May 2026, ai_autonomy_mean",
    "Original": f"value={anthropic_original['value']}",
    "Expected": "Retained; value=2.47",
    "Actual": f"Retained; value={anthropic_actual['value']}",
    "Match": anthropic_actual["value"] == 2.47,
})

# 2. BLS unavailable employment
bls_emp_original = msa_raw[
    (msa_raw["AREA"] == "12540")
    & (msa_raw["OCC_CODE"] == "13-2082")
].iloc[0]

bls_emp_actual = bls_clean[
    (bls_clean["AREA"] == "12540")
    & (bls_clean["OCC_CODE"] == "13-2082")
].iloc[0]

record_checks.append({
    "Check": 2,
    "Record": "Bakersfield 13-2082 Tax Preparers",
    "Original": f"TOT_EMP={bls_emp_original['TOT_EMP']}",
    "Expected": "TOT_EMP=missing; unavailable_flag=True",
    "Actual": (
        f"TOT_EMP={'missing' if pd.isna(bls_emp_actual['TOT_EMP']) else bls_emp_actual['TOT_EMP']}; "
        f"unavailable_flag={bls_emp_actual['TOT_EMP_unavailable_flag']}"
    ),
    "Match": (
        pd.isna(bls_emp_actual["TOT_EMP"])
        and bool(bls_emp_actual["TOT_EMP_unavailable_flag"])
    ),
})

# 3. BLS high-wage threshold
bls_wage_original = msa_raw[
    (msa_raw["AREA"] == "12540")
    & (msa_raw["OCC_CODE"] == "29-1214")
].iloc[0]

bls_wage_actual = bls_clean[
    (bls_clean["AREA"] == "12540")
    & (bls_clean["OCC_CODE"] == "29-1214")
].iloc[0]

record_checks.append({
    "Check": 3,
    "Record": "Bakersfield 29-1214 Emergency Medicine Physicians",
    "Original": f"A_MEDIAN={bls_wage_original['A_MEDIAN']}",
    "Expected": "A_MEDIAN=missing; high_wage_flag=True",
    "Actual": (
        f"A_MEDIAN={'missing' if pd.isna(bls_wage_actual['A_MEDIAN']) else bls_wage_actual['A_MEDIAN']}; "
        f"high_wage_flag={bls_wage_actual['A_MEDIAN_high_wage_flag']}"
    ),
    "Match": (
        pd.isna(bls_wage_actual["A_MEDIAN"])
        and bool(bls_wage_actual["A_MEDIAN_high_wage_flag"])
    ),
})

# 4. O*NET SOC6 derivation
onet_original = onet_raw[
    onet_raw["O*NET-SOC Code"] == "11-1011.03"
].iloc[0]

onet_actual = onet_clean[
    onet_clean["O*NET-SOC Code"] == "11-1011.03"
].iloc[0]

record_checks.append({
    "Check": 4,
    "Record": "O*NET 11-1011.03 Chief Sustainability Officers",
    "Original": f"O*NET-SOC Code={onet_original['O*NET-SOC Code']}",
    "Expected": "Full code=11-1011.03; SOC6=11-1011",
    "Actual": (
        f"Full code={onet_actual['O*NET-SOC Code']}; "
        f"SOC6={onet_actual['SOC6']}"
    ),
    "Match": (
        onet_actual["O*NET-SOC Code"] == "11-1011.03"
        and onet_actual["SOC6"] == "11-1011"
    ),
})

# 5. Unmatched BLS occupation
merge_actual = bls_onet[
    (bls_onet["AREA"] == "12540")
    & (bls_onet["OCC_CODE"] == "13-1020")
].iloc[0]

record_checks.append({
    "Check": 5,
    "Record": "Bakersfield 13-1020 Buyers and Purchasing Agents",
    "Original": "BLS OCC_CODE=13-1020",
    "Expected": "Retained; O*NET direct match status=left_only",
    "Actual": f"Retained; match status={merge_actual['_bls_onet_merge']}",
    "Match": merge_actual["_bls_onet_merge"] == "left_only",
})

record_check_table = pd.DataFrame(record_checks)
record_check_table


,Check,Record,Original,Expected,Actual,Match
0,1,"Anthropic 49-3092.00, May 2026, ai_autonomy_mean",value=2.47,Retained; value=2.47,Retained; value=2.47,True
1,2,Bakersfield 13-2082 Tax Preparers,TOT_EMP=**,TOT_EMP=missing; unavailable_flag=True,TOT_EMP=missing; unavailable_flag=True,True
2,3,Bakersfield 29-1214 Emergency Medicine Physicians,A_MEDIAN=#,A_MEDIAN=missing; high_wage_flag=True,A_MEDIAN=missing; high_wage_flag=True,True
3,4,O*NET 11-1011.03 Chief Sustainability Officers,O*NET-SOC Code=11-1011.03,Full code=11-1011.03; SOC6=11-1011,Full code=11-1011.03; SOC6=11-1011,True
4,5,Bakersfield 13-1020 Buyers and Purchasing Agents,BLS OCC_CODE=13-1020,Retained; O*NET direct match status=left_only,Retained; match status=left_only,True


In [91]:
assert record_check_table["Match"].all()

print("All five record checks matched the expected results.")


All five record checks matched the expected results.


## 7. Save Cleaned Outputs

Cleaned and integrated outputs are written only after all cleaning and
verification steps have completed successfully.

In [92]:
from pathlib import Path

output_dir = Path("cleaned_output")
output_dir.mkdir(exist_ok=True)

In [93]:
anthropic_clean.to_csv(
    output_dir / "anthropic_clean.csv",
    index=False,
)

bls_clean.to_csv(
    output_dir / "bls_clean.csv",
    index=False,
)

onet_clean.to_csv(
    output_dir / "onet_clean.csv",
    index=False,
)

In [94]:
integrated_long.to_csv(
    output_dir / "integrated_long.csv",
    index=False,
)

## 8. Cleaned Data Sample

A sample of 50 cleaned integrated rows is selected deliberately rather than
randomly. The sample includes both typical observations and difficult cases:

- 10 normal matched metropolitan records;
- 10 normal matched nonmetropolitan records;
- 10 records with unavailable BLS employment;
- 10 records with BLS high-wage flags; and
- 10 BLS records without an Anthropic match.

Only a concise set of analysis and audit fields is included in the submitted
sample.


In [95]:
sample_cases = [
    (
        "normal_metro",
        (integrated_long["area_type"] == "metro")
        & (integrated_long["_bls_anthropic_merge"] == "both")
        & integrated_long["TOT_EMP"].notna()
        & integrated_long["A_MEDIAN"].notna()
        & ~integrated_long["TOT_EMP_unavailable_flag"]
        & ~integrated_long["A_MEDIAN_high_wage_flag"],
    ),
    (
        "normal_nonmetro",
        (integrated_long["area_type"] == "nonmetro")
        & (integrated_long["_bls_anthropic_merge"] == "both")
        & integrated_long["TOT_EMP"].notna()
        & integrated_long["A_MEDIAN"].notna()
        & ~integrated_long["TOT_EMP_unavailable_flag"]
        & ~integrated_long["A_MEDIAN_high_wage_flag"],
    ),
    (
        "unavailable_employment",
        integrated_long["TOT_EMP_unavailable_flag"],
    ),
    (
        "high_wage",
        integrated_long["A_MEDIAN_high_wage_flag"],
    ),
    (
        "no_anthropic_match",
        integrated_long["_bls_anthropic_merge"] == "left_only",
    ),
]

selected_parts = []
used_keys = set()

for label, mask in sample_cases:
    candidates = (
        integrated_long.loc[mask]
        .copy()
        .drop_duplicates(subset=["AREA", "OCC_CODE"])
    )

    candidate_keys = list(
        zip(candidates["AREA"], candidates["OCC_CODE"])
    )
    candidates = candidates[
        [key not in used_keys for key in candidate_keys]
    ].head(10).copy()

    candidates["sample_case"] = label

    used_keys.update(
        zip(candidates["AREA"], candidates["OCC_CODE"])
    )

    selected_parts.append(candidates)

cleaned_sample = pd.concat(
    selected_parts,
    ignore_index=True,
)

sample_columns = [
    "sample_case",
    "AREA",
    "AREA_TITLE",
    "area_type",
    "OCC_CODE",
    "OCC_TITLE",
    "TOT_EMP",
    "TOT_EMP_unavailable_flag",
    "A_MEDIAN",
    "A_MEDIAN_high_wage_flag",
    "O*NET-SOC Code_bls",
    "onet_title_bls",
    "date_start",
    "date_end",
    "geo_level",
    "metric_id",
    "value",
    "_bls_onet_merge",
    "_bls_anthropic_merge",
]

cleaned_sample = cleaned_sample[
    sample_columns
]

cleaned_sample["sample_case"].value_counts()


sample_case
normal_metro              10
normal_nonmetro           10
unavailable_employment    10
high_wage                 10
no_anthropic_match        10
Name: count, dtype: int64

In [96]:
cleaned_sample


,sample_case,AREA,AREA_TITLE,area_type,OCC_CODE,OCC_TITLE,TOT_EMP,TOT_EMP_unavailable_flag,A_MEDIAN,A_MEDIAN_high_wage_flag,O*NET-SOC Code_bls,onet_title_bls,date_start,date_end,geo_level,metric_id,value,_bls_onet_merge,_bls_anthropic_merge
0,normal_metro,12540,"Bakersfield-Delano, CA",metro,11-1011,Chief Executives,390.0,False,195400.0,False,11-1011.00,Chief Executives,2026-05-01,2026-06-01,global,ai_autonomy_mean,2.91,both,both
1,normal_metro,12540,"Bakersfield-Delano, CA",metro,11-1021,General and Operations Managers,4100.0,False,104460.0,False,11-1021.00,General and Operations Managers,2026-05-01,2026-06-01,global,ai_autonomy_mean,2.97,both,both
2,normal_metro,12540,"Bakersfield-Delano, CA",metro,11-2021,Marketing Managers,240.0,False,149960.0,False,11-2021.00,Marketing Managers,2026-05-01,2026-06-01,global,ai_autonomy_mean,2.94,both,both
3,normal_metro,12540,"Bakersfield-Delano, CA",metro,11-2022,Sales Managers,1220.0,False,98270.0,False,11-2022.00,Sales Managers,2026-05-01,2026-06-01,global,ai_autonomy_mean,2.70,both,both
4,normal_metro,12540,"Bakersfield-Delano, CA",metro,11-2032,Public Relations Managers,40.0,False,137200.0,False,11-2032.00,Public Relations Managers,2026-05-01,2026-06-01,global,ai_autonomy_mean,3.05,both,both
5,normal_metro,12540,"Bakersfield-Delano, CA",metro,11-2033,Fundraising Managers,40.0,False,130340.0,False,11-2033.00,Fundraising Managers,2026-05-01,2026-06-01,global,ai_autonomy_mean,3.12,both,both
6,normal_metro,12540,"Bakersfield-Delano, CA",metro,11-3012,Administrative Services Managers,490.0,False,101780.0,False,11-3012.00,Administrative Services Managers,2026-04-01,2026-05-01,global,ai_autonomy_mean,2.82,both,both
7,normal_metro,12540,"Bakersfield-Delano, CA",metro,11-3013,Facilities Managers,310.0,False,107190.0,False,11-3013.00,Facilities Managers,2026-05-01,2026-06-01,global,ai_autonomy_mean,3.16,both,both
8,normal_metro,12540,"Bakersfield-Delano, CA",metro,11-3021,Computer and Information Systems Managers,430.0,False,164470.0,False,11-3021.00,Computer and Information Systems Managers,2026-05-01,2026-06-01,global,ai_autonomy_mean,2.66,both,both
9,normal_metro,12540,"Bakersfield-Delano, CA",metro,11-3031,Financial Managers,900.0,False,153030.0,False,11-3031.00,Financial Managers,2026-05-01,2026-06-01,global,ai_autonomy_mean,2.90,both,both


In [97]:
assert len(cleaned_sample) <= 50
assert cleaned_sample["sample_case"].nunique() == 5

cleaned_sample.to_csv(
    output_dir / "cleaned_sample.csv",
    index=False,
)

print(f"Saved {len(cleaned_sample)} cleaned sample rows.")


Saved 50 cleaned sample rows.
